# Imports

In [1]:
import os
import numpy as np
import pandas as pd


# Definicion de constantes

In [2]:
# Rutas
RUTA_LECTURA = "../Data/BANK_marketing_raw.csv"
RUTA_ESCRITURA = "../Data/BANK_marketing_clean.csv"

# Carga de datos

In [3]:
df_bank_marketing = pd.read_csv(RUTA_LECTURA)

# DATA CLEANING

## Comprobación y eliminación de duplicados

* 1. Comprobar duplicados por ID único
* 2. Comprobar si hay filas exactamente iguales (sin contar el ID)
* 3. Limpieza (en caso de que existan, elimina manteniendo la primera aparición)

In [4]:

duplicados_id = df_bank_marketing.duplicated(subset=['id']).sum()
print(f"Filas con ID duplicado: {duplicados_id}")


columnas_sin_id = [col for col in df_bank_marketing.columns if col != 'id']
duplicados_filas = df_bank_marketing.duplicated(subset=columnas_sin_id).sum()
print(f"Filas duplicadas en contenido (ignorando ID): {duplicados_filas}")


if duplicados_filas > 0:
    df_bank_marketing = df_bank_marketing.drop_duplicates(subset=columnas_sin_id, keep='first')

Filas con ID duplicado: 0
Filas duplicadas en contenido (ignorando ID): 0


## Tratamiento de valores faltantes (Null y Unknown)

* 1. Imputar numéricos ('age') con la mediana
* 2. Imputar categóricos ('marital', 'education', 'job') con la moda (para los nulls de marital y education, y los unknown de education y job)
* 3. Manejo de "unknown" estratégicos
    * 'contact' (20%) se quedan como "unknown" porque ya es una categoría.
    * Renombrar el "unknown" de 'poutcome' a su significado de negocio 'no_pcampaign'.

In [5]:
mediana_edad = df_bank_marketing['age'].median()
df_bank_marketing['age'] = df_bank_marketing['age'].fillna(mediana_edad)


moda_marital = df_bank_marketing['marital'].mode()[0]
df_bank_marketing['marital'] = df_bank_marketing['marital'].fillna(moda_marital)

moda_education = df_bank_marketing['education'].mode()[0]
df_bank_marketing['education'] = df_bank_marketing['education'].fillna(moda_education)
df_bank_marketing['education'] = df_bank_marketing['education'].replace('unknown', moda_education)

moda_job = df_bank_marketing[df_bank_marketing['job'] != 'unknown']['job'].mode()[0]
df_bank_marketing['job'] = df_bank_marketing['job'].replace('unknown', moda_job)


df_bank_marketing['poutcome'] = df_bank_marketing['poutcome'].replace('unknown', 'no_pcampaign')


## Estandarización de formatos y tipos de datos

* 1. Estandarizar textos a minúsculas y sin espacios
* 2. Mapear todas las variables binarias de yes/no a 1/0 enteros (dejamos el dataset nativamente listo para cualquier algoritmo estadístico sin necesidad de hacer pasos extra más adelante.)
* 3. Corrección de tipos numéricos (asegurar enteros)
* 4. Convertir columnas categóricas multi-clase a tipo 'category' (las columnas de texto (object) consumen mucha memoria porque guardan cada palabra textualmente. El tipo category funciona como un diccionario: asigna un número interno (ej. Married = 1, Single = 2) pero nos sigue mostrando el texto)

In [6]:
columnas_texto = df_bank_marketing.select_dtypes(include=['object']).columns
for col in columnas_texto:
    df_bank_marketing[col] = df_bank_marketing[col].astype(str).str.strip().str.lower()


columnas_binarias = ['deposit', 'default', 'housing', 'loan']
for col in columnas_binarias:
    df_bank_marketing[col] = df_bank_marketing[col].map({'yes': 1, 'no': 0})


df_bank_marketing['age'] = df_bank_marketing['age'].astype(int)

columnas_a_categoria = ['job', 'marital', 'education', 'contact', 'month', 'poutcome']
for col in columnas_a_categoria:
    if col in df_bank_marketing.columns:
        df_bank_marketing[col] = df_bank_marketing[col].astype('category')


C:\Users\raari\AppData\Local\Temp\ipykernel_37908\522589309.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_texto = df_bank_marketing.select_dtypes(include=['object']).columns


## Verificación final del estado del Dataframe

In [7]:
print(f"Final Dataframe Shape: {df_bank_marketing.shape}")
print("\nMissing values check:")
print(df_bank_marketing.isnull().sum())
print("\nOptimized Data Types:")
print(df_bank_marketing.dtypes)
print("\nUnique values in poutcome:")
print(df_bank_marketing['poutcome'].unique())


Final Dataframe Shape: (10642, 18)

Missing values check:
id           0
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
deposit      0
dtype: int64

Optimized Data Types:
id              int64
age             int64
job          category
marital      category
education    category
default         int64
balance         int64
housing         int64
loan            int64
contact      category
day             int64
month        category
duration        int64
campaign        int64
pdays           int64
previous        int64
poutcome     category
deposit         int64
dtype: object

Unique values in poutcome:
['no_pcampaign', 'other', 'failure', 'success']
Categories (4, str): ['failure', 'no_pcampaign', 'other', 'success']


## Guardar el DataFrame limpio en un archivo CSV

In [8]:
df_bank_marketing.to_csv(RUTA_ESCRITURA, index=False)